In [ ]:
import os
import csv
import argparse
import codecs
import re
import logging

import numpy as np
import scipy.io as sio
from natsort import natsorted
from scipy.signal import butter, sosfiltfilt

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

In [ ]:
# -------------------------------------------------
# 带宽滤波
# -------------------------------------------------
def bandpass(signal, fs, low=5.0, high=300.0, order=4):
    """
    Apply a Butterworth bandpass (zero-phase via sosfiltfilt) to a 1D signal.
    """
    sos = butter(order, [low, high], btype='band', fs=fs, output='sos')
    return sosfiltfilt(sos, signal)

# -------------------------------------------------
# 解析 label txt
# -------------------------------------------------
def parse_label_txt(label_path):

    info = {
        "class": None,
        "channel": None,
        "intensity": None,
        "env": ""
    }

    try:
        with codecs.open(label_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:

                line = line.strip()

                if not line:
                    continue

                # 前三行都有 =
                if "=" in line:

                    key, val = line.split("=", 1)

                    key = key.lower()

                    if "data_path" in key:
                        info["class"] = val.strip()

                    elif "channel" in key:
                        info["channel"] = int(val)

                    elif "intensity" in key:
                        info["intensity"] = int(val)

                # 第四行（环境描述）
                else:
                    parts = line.split()
                    if parts:
                        info["env"] = parts[-1]

    except Exception as e:
        logging.exception("读取label失败: %s | %s", label_path, e)

    return info

# -------------------------------------------------
# 找 label
# -------------------------------------------------
def find_label_for_event_and_parse(label_root, class_name, session_folder):

    candidate_dir = os.path.join(label_root, class_name)

    if not os.path.isdir(candidate_dir):
        return None, None

    s1 = re.sub(r'[^0-9]', '', session_folder)

    for fn in os.listdir(candidate_dir):
        name, ext = os.path.splitext(fn)
        s2 = re.sub(r'[^0-9]', '', name)

        if s1 in s2 or s2 in s1:

            full = os.path.join(candidate_dir, fn)

            if ext.lower() == ".txt":

                parsed = parse_label_txt(full)

                return full, parsed

    logging.debug("No label found in %s for session %s", candidate_dir, session_folder)
    return None, None

# -------------------------------------------------
# 读取 mat
# -------------------------------------------------
def load_and_combine_mat_files(session_path):
    """
    Load all .mat files in session_path, extract variable 's' from each, and concatenate along time axis.
    Expectation: each mat['s'] is an array of shape (ch, samples).
    """

    mat_files = [f for f in os.listdir(session_path) if f.endswith(".mat")]

    mat_files = natsorted(mat_files)

    arrays = []

    for f in mat_files:

        file_path = os.path.join(session_path, f)

        try:

            mat = sio.loadmat(file_path)

            if "s" not in mat:
                logging.debug("MAT file %s does not contain 's', skipping", file_path)
                continue

            arr = mat["s"]
            if arr.ndim == 1:
                arr = arr[np.newaxis, :] # Ensure 2D
            arrays.append(arr)

        except Exception as e:
            logging.exception("读取失败: %s | %s", file_path, e)

    if not arrays:
        return None

    combined = np.concatenate(arrays, axis=1)

    return combined

# -------------------------------------------------
# 保存窗口
# -------------------------------------------------
def save_windows_and_metadata(
        combined_arr,
        out_dir,
        class_name,
        session_folder,
        window_len,
        hop,
        label_parsed,
        meta_writer
):

    ch, total = combined_arr.shape

    os.makedirs(out_dir, exist_ok=True)

    idx = 0

    for start in range(0, total - window_len + 1, hop):

        win = combined_arr[:, start:start + window_len]

        save_arr = win.T.astype(np.float32)

        fname = f"{class_name}__{session_folder}__{idx:06d}.npy"

        path = os.path.join(out_dir, fname)

        np.save(path, save_arr)

        cls = label_parsed.get("class") if label_parsed else ""

        chn = label_parsed.get("channel") if label_parsed else ""

        inten = label_parsed.get("intensity") if label_parsed else ""

        env = label_parsed.get("env") if label_parsed else ""

        meta_writer.writerow([
            fname,
            class_name,
            session_folder,
            start,
            start + window_len,
            cls,
            chn,
            inten,
            env
        ])

        idx += 1

    return idx
    # -------------------------------------------------
# 主函数
# -------------------------------------------------
def main(args):

    data_root = args.data_root
    label_root = args.label_root
    out_root = args.out_root

    window_len = args.window_len
    hop = args.hop

    fs = args.fs
    low = args.low
    high = args.high
    order = args.order
    apply_bp = args.apply_bandpass

    os.makedirs(out_root, exist_ok=True)

    metadata_path = os.path.join(out_root, "metadata.csv")

    with open(metadata_path, "w", newline='', encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "filename",
            "class",
            "session_folder",
            "start",
            "end",
            "label_class",
            "label_channel",
            "label_intensity",
            "label_env"
        ])

        for class_name in os.listdir(data_root):

            class_dir = os.path.join(data_root, class_name)

            if not os.path.isdir(class_dir):
                continue

            logging.info("处理类别: %s", class_name)

            out_class_dir = os.path.join(out_root, class_name)

            for session_folder in os.listdir(class_dir):

                session_path = os.path.join(class_dir, session_folder)

                if not os.path.isdir(session_path):
                    continue

                logging.info("  session: %s", session_folder)

                combined = load_and_combine_mat_files(session_path)

                if combined is None:
                    logging.warning("  无有效数据 for session %s", session_folder)
                    continue

                # apply bandpass to each channel of combined MAT array if requested
                if apply_bp:
                    logging.info("  Applying bandpass to combined array: %s - %s Hz, fs=%s, order=%s",
                                 low, high, fs, order)
                    try:
                        # combined shape is (channels, samples)
                        for c in range(combined.shape[0]):
                            combined[c, :] = bandpass(combined[c, :].astype(np.float64), fs=fs, low=low, high=high, order=order)
                    except Exception as e:
                        logging.exception("  Bandpass failed for session %s: %s", session_folder, e)
                        continue

                label_path, label_parsed = find_label_for_event_and_parse(
                    label_root,
                    class_name,
                    session_folder
                )

                save_windows_and_metadata(
                    combined,
                    out_class_dir,
                    class_name,
                    session_folder,
                    window_len,
                    hop,
                    label_parsed,
                    writer
                )

    logging.info("预处理完成")
    logging.info("metadata: %s", metadata_path)
# -------------------------------------------------
# CLI
# -------------------------------------------------
# if __name__ == "__main__":

#     parser = argparse.ArgumentParser()

#     parser.add_argument("--window_len", type=int, default=2048, help="window length in samples (default 2048)")
    # parser.add_argument("--hop", type=int, default=512, help="hop/stride in samples (default 512)")

    # --- bandpass params added for MAT files
    # parser.add_argument("--fs", type=float, default=1000.0, help="sampling rate in Hz for bandpass (must match your MAT data)")
    # parser.add_argument("--low", type=float, default=5.0, help="bandpass low cutoff (Hz)")
    # parser.add_argument("--high", type=float, default=300.0, help="bandpass high cutoff (Hz)")
    # parser.add_argument("--order", type=int, default=4, help="Butterworth filter order")
    # parser.add_argument("--apply_bandpass", action="store_true", help="apply bandpass to mat channels before windowing")
#     args = parser.parse_args()

#     main(args)

In [ ]:
args = argparse.Namespace(
            data_root="/home/sente/das_ai_project/data/train/data",
            label_root="/home/sente/das_ai_project/data/train/label",
            out_root="/home/sente/das_ai_project/data/train/processed_data",
            window_len=100,
            hop=50
)
main(args)

处理类别: pawang
  session: 2021-10-22 14_41_03
  session: 2021-10-22 15_12_21
  session: 2021-10-22 14_51_38
  session: 2021-10-22 14_37_20
  session: 2021-10-22 14_58_22
  session: 2021-10-22 15_01_31
  session: 2021-10-22 14_47_59
  session: 2021-10-22 14_54_57
  session: 2021-10-22 15_04_31
  session: 2021-10-22 14_44_37
处理类别: zuanwang
  session: 2021-10-22 14_10_07
  session: 2021-10-22 13_43_20
  session: 2021-10-22 14_13_42
  session: 2021-10-22 13_53_03
  session: 2021-10-22 14_16_34
  session: 2021-10-22 14_02_39
  session: 2021-10-22 13_55_58
  session: 2021-10-22 13_45_48
  session: 2021-10-22 13_59_01
  session: 2021-10-22 14_06_07
预处理完成
metadata: /home/sente/das_ai_project/data/train/processed_data/metadata.csv


In [2]:
import numpy as np
import pandas as pd

# 读取两个结果
d1 = np.load("/home/sente/das_ai_project/data/psd_data_pawang.npz")
d2 = np.load("/home/sente/das_ai_project/data/psd_data_zuanwang.npz")

f = d1["f"]
psd1, psd2 = d1["psd"], d2["psd"]

# 做差
diff = psd1 - psd2
abs_diff = np.abs(diff)

# 最大差
mask = f > 1
idx = np.argmax(abs_diff[mask])
print("最大差:", abs_diff[mask][idx], "频率:", f[mask][idx])

# 保存成 Excel 可读格式（CSV）
pd.DataFrame({
    "freq": f,
    "psd1": psd1,
    "psd2": psd2,
    "abs_diff": abs_diff
}).to_csv("/home/sente/das_ai_project/data/psd_compare.csv", index=False)

最大差: 0.028883338 频率: 2.63671875
